In [1]:
# Install libraries FIRST
!pip install apache-sedona==1.6.1
!sudo yum install -y java-1.8.0-openjdk-devel

  Installing build dependencies ... done
  Getting requirements to build wheel ... error
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [2 lines of output]
      ERROR: A GDAL API version must be specified. Provide a path to gdal-config using a GDAL_CONFIG environment variable or use a GDAL_VERSION environment variable.
      [end of output]
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
Loaded plugins: dkms-build-requires, extras_suggestions, kernel-livepatch,
              : langpacks, priorities, update-motd, versionlock
amzn2-core                                               | 3.6 kB     00:00     
amzn2extra-docker                  

In [2]:
import time
import csv
import io
from pyspark.sql.functions import col, when
from project_setup import get_spark_session, CRIME_DATA

spark = get_spark_session("Query1_Analysis") #default: executors="4", cores="1", memory="2g"
sc = spark.sparkContext

Configuring Environment for 'Query1_Analysis'...
   Resource Config: 4 Executors | 1 Cores | 2g RAM
   JAVA_HOME set to: /usr/lib/jvm/java-1.8.0-openjdk-1.8.0.472.b08-1.amzn2.0.1.x86_64/jre
:: loading settings :: url = jar:file:/home/ec2-user/anaconda3/envs/python3/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/ec2-user/.ivy2/cache
The jars for the packages stored in: /home/ec2-user/.ivy2/jars
org.apache.sedona#sedona-spark-shaded-3.4_2.12 added as a dependency
org.datasyslab#geotools-wrapper added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-920b470a-16c4-48ca-8141-84b136df5b8e;1.0
	confs: [default]
	found org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1 in central
	found org.datasyslab#geotools-wrapper;1.6.1-28.2 in central
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found com.amazonaws#aws-java-sdk-bundle;1.11.1026 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
downloading https://repo1.maven.org/maven2/org/apache/sedona/sedona-spark-shaded-3.4_2.12/1.6.1/sedona-spark-shaded-3.4_2.12-1.6.1.jar ...
	[SUCCESSFUL ] org.apache.sedona#sedona-spark-shaded-3.4_2.12;1.6.1!sedona-spark-shaded-3.4_2.12.jar (672ms)
downloading https://repo

25/12/14 22:26:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


   Sedona library not found.


In [4]:
#Load Data & Preview Columns
# Φόρτωση πρώτου αρχείου (2010-2019)
print(f"--- Loading: {CRIME_DATA[0].split('/')[-1]} ---")
df = spark.read.option("header", "true").option("inferSchema", "true").csv(CRIME_DATA[0])
print(f"Columns: {df.columns}")  

# Φόρτωση επόμενων αρχείων (2020-) και Union
for path in CRIME_DATA[1:]:
    print(f"--- Loading: {path.split('/')[-1]} ---")
    df_new = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
    print(f"Columns: {df_new.columns}")  
    df = df.union(df_new)

--- Loading: LA_Crime_Data_2010_2019.csv ---


Columns: ['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME', 'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes', 'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc', 'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1', 'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT', 'LON']
--- Loading: LA_Crime_Data_2020_2025.csv ---


[Stage 9:=======================================>                   (2 + 1) / 3]

Columns: ['DR_NO', 'Date Rptd', 'DATE OCC', 'TIME OCC', 'AREA', 'AREA NAME', 'Rpt Dist No', 'Part 1-2', 'Crm Cd', 'Crm Cd Desc', 'Mocodes', 'Vict Age', 'Vict Sex', 'Vict Descent', 'Premis Cd', 'Premis Desc', 'Weapon Used Cd', 'Weapon Desc', 'Status', 'Status Desc', 'Crm Cd 1', 'Crm Cd 2', 'Crm Cd 3', 'Crm Cd 4', 'LOCATION', 'Cross Street', 'LAT', 'LON']


In [7]:
df.select("Crm Cd Desc", "Vict Age").show(10, truncate=False)

+--------------------------------------------------------+--------+
|Crm Cd Desc                                             |Vict Age|
+--------------------------------------------------------+--------+
|VIOLATION OF COURT ORDER                                |48      |
|VANDALISM - FELONY ($400 & OVER, ALL CHURCH VANDALISMS) |0       |
|OTHER MISCELLANEOUS CRIME                               |0       |
|VIOLATION OF COURT ORDER                                |47      |
|RAPE, ATTEMPTED                                         |47      |
|SHOPLIFTING - PETTY THEFT ($950 & UNDER)                |23      |
|BURGLARY FROM VEHICLE                                   |46      |
|ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT          |51      |
|ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT          |30      |
|THEFT-GRAND ($950.01 & OVER)EXCPT,GUNS,FOWL,LIVESTK,PROD|55      |
+--------------------------------------------------------+--------+
only showing top 10 rows



In [11]:
print("Starting Query 1 (DataFrame API)...")
start_time = time.time()

print(f"--- Loading: {CRIME_DATA[0].split('/')[-1]} ---")
df = spark.read.option("header", "true").option("inferSchema", "true").csv(CRIME_DATA[0])

for path in CRIME_DATA[1:]:
    print(f"--- Loading: {path.split('/')[-1]} ---")
    df_new = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
    df = df.union(df_new)

filtered_df = df.filter(col("Crm Cd Desc").contains("AGGRAVATED ASSAULT"))

#Age Groups 
df_grouped = filtered_df.withColumn("AgeGroup", 
    when(col("Vict Age") < 18, "Παιδιά")
    .when((col("Vict Age") >= 18) & (col("Vict Age") <= 24), "Νεαροί ενήλικοι")
    .when((col("Vict Age") >= 25) & (col("Vict Age") <= 64), "Ενήλικοι")
    .when(col("Vict Age") > 64, "Ηλικιωμένοι")
    .otherwise("Unknown")
)

#Aggregate
result = df_grouped.filter(col("AgeGroup") != "Unknown") \
                   .groupBy("AgeGroup") \
                   .count() \
                   .orderBy(col("count").desc())

result.show()
print(f"Time: {time.time() - start_time:.2f}s")

Starting Query 1 (DataFrame API)...
--- Loading: LA_Crime_Data_2010_2019.csv ---


--- Loading: LA_Crime_Data_2020_2025.csv ---


[Stage 26:==================================================>       (7 + 1) / 8]

+---------------+------+
|       AgeGroup| count|
+---------------+------+
|       Ενήλικοι|121660|
|Νεαροί ενήλικοι| 33758|
|         Παιδιά| 16014|
|    Ηλικιωμένοι|  6011|
+---------------+------+

Time: 30.44s


In [12]:
from pyspark.sql.functions import col, udf
from pyspark.sql.types import StringType

print("Starting Query 1 (UDF Approach)...")

def categorize_age_python(age):
    if age is None:
        return "Unknown"
    try:
        age = int(age)
    except:
        return "Unknown"

    if age < 18:
        return "Παιδιά"
    elif 18 <= age <= 24:
        return "Νεαροί ενήλικοι"
    elif 25 <= age <= 64:
        return "Ενήλικοι"
    elif age > 64:
        return "Ηλικιωμένοι"
    else:
        return "Unknown"

# --- Εγγραφή της UDF ---
age_udf = udf(categorize_age_python, StringType())

start_time = time.time()

print(f"--- Loading: {CRIME_DATA[0].split('/')[-1]} ---")
df = spark.read.option("header", "true").option("inferSchema", "true").csv(CRIME_DATA[0])

for path in CRIME_DATA[1:]:
    print(f"--- Loading: {path.split('/')[-1]} ---")
    df_new = spark.read.option("header", "true").option("inferSchema", "true").csv(path)
    df = df.union(df_new)

filtered_df = df.filter(col("Crm Cd Desc").contains("AGGRAVATED ASSAULT"))

df_grouped_udf = filtered_df.withColumn("AgeGroup", age_udf(col("Vict Age")))

# Aggregation 
result_udf = df_grouped_udf.filter(col("AgeGroup") != "Unknown") \
                           .groupBy("AgeGroup") \
                           .count() \
                           .orderBy(col("count").desc())

result_udf.show()
print(f"Time (UDF): {time.time() - start_time:.2f}s")

Starting Query 1 (UDF Approach)...
--- Loading: LA_Crime_Data_2010_2019.csv ---


--- Loading: LA_Crime_Data_2020_2025.csv ---


[Stage 33:==================================================>       (7 + 1) / 8]

+---------------+------+
|       AgeGroup| count|
+---------------+------+
|       Ενήλικοι|121660|
|Νεαροί ενήλικοι| 33758|
|         Παιδιά| 16014|
|    Ηλικιωμένοι|  6011|
+---------------+------+

Time (UDF): 33.18s


In [13]:
# --- RDD API IMPLEMENTATION ---
print("Starting Query 1 (RDD API)...")
start_time = time.time()

rdd = sc.textFile(",".join(CRIME_DATA))
header_line = rdd.first()

reader = csv.reader(io.StringIO(header_line))
header_columns = next(reader)
try:
    age_index = header_columns.index("Vict Age")
    desc_index = header_columns.index("Crm Cd Desc")
except ValueError:
    age_index, desc_index = 11, 8

def process_line(line):
    if line == header_line: return None
    try:
        reader = csv.reader(io.StringIO(line))
        fields = next(reader)
        if len(fields) <= max(age_index, desc_index): return None
        
        description = fields[desc_index]
        if "AGGRAVATED ASSAULT" not in description: return None
        
        age_str = fields[age_index]
        if not age_str.strip().lstrip('-').isdigit(): return ("Unknown", 1)
        
        age = int(age_str)
        if age < 0: return ("Unknown", 1)
        if age < 18: group = "Παιδιά"
        elif 18 <= age <= 24: group = "Νεαροί ενήλικοι"
        elif 25 <= age <= 64: group = "Ενήλικοι"
        else: group = "Ηλικιωμένοι"
        
        return (group, 1)
    except: return None

result_rdd = rdd.map(process_line) \
                .filter(lambda x: x is not None) \
                .reduceByKey(lambda a, b: a + b) \
                .sortBy(lambda x: x[1], ascending=False)

print("\nRDD Results:")
print(result_rdd.collect())
print(f"RDD Execution Time: {time.time() - start_time:.2f} seconds")

Starting Query 1 (RDD API)...



RDD Results:


[Stage 43:================================================>       (24 + 2) / 28]

[('Ενήλικοι', 121660), ('Νεαροί ενήλικοι', 33758), ('Παιδιά', 16009), ('Ηλικιωμένοι', 6011), ('Unknown', 5)]
RDD Execution Time: 41.76 seconds
